# Laboratorio práctico de ETL — Kaismart Solutions S.A.S.

Maestría en Ciencia de Datos e IA · Universidad Autónoma de Occidente (UAO)

Este cuaderno desarrolla el proceso de extracción de las dos fuentes de información de la empresa:

| Fuente | Origen | Registros esperados |
|---|---|---|
| Sistema comercial | Base de datos MySQL `clientes`, tabla `ventas` | 5.000 |
| Sistema logístico | Archivo Excel `kaismart_eventos_logisticos.xlsx` | 50.000 |

**Alcance de esta entrega:** Parte 1 (extracción desde MySQL) y Parte 2 (extracción desde Excel).
En estas partes **no se convierten tipos, no se limpia y no se imputa nada**: los datos se guardan tal como llegan.

## Arquitectura Medallion

| Capa | Contenido | Carpeta | Se usa en |
|---|---|---|---|
| **Bronze** | Datos originales, tal como se extrajeron de la fuente, sin transformar | `data/bronze/` | Partes 1 y 2 |
| **Silver** | Datos depurados: nulos tratados, duplicados eliminados, categorías estandarizadas, tipos corregidos | `data/silver/` | Parte 7 |
| **Gold** | Datos integrados (ventas + logística por `pedido_id`) listos para análisis | `data/gold/` | Parte 7 |

## Configuración

Se carga `config.yaml` (fuentes, rutas y autores) y el archivo `.env` (contraseña de la base de datos, para que no quede escrita en el código).
Las rutas son relativas a la raíz del proyecto.

In [1]:
from pathlib import Path
import os

import pandas as pd
import yaml
from dotenv import load_dotenv

# Raíz del proyecto: la carpeta desde la que se ejecuta el cuaderno
RAIZ = Path.cwd()
assert (RAIZ / "config.yaml").exists(), f"No se encontró config.yaml en {RAIZ}"

# Configuración general (config.yaml) y contraseña (.env)
with open(RAIZ / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
load_dotenv(RAIZ / ".env")

# Rutas de las capas; se crean si no existen
rutas = {nombre: RAIZ / ruta for nombre, ruta in config["rutas"].items()}
for ruta in rutas.values():
    ruta.mkdir(parents=True, exist_ok=True)

print("Autores:", ", ".join(config["autores"]))
print("Raíz del proyecto:", RAIZ)
for nombre, ruta in rutas.items():
    print(f"  {nombre:<7}-> {ruta}")

Autores: Simón Colonia, Integrante 2
Raíz del proyecto: C:\Users\simon\Documents\lab2etl\kaismart_etl
  raw    -> C:\Users\simon\Documents\lab2etl\kaismart_etl\data\raw
  bronze -> C:\Users\simon\Documents\lab2etl\kaismart_etl\data\bronze
  silver -> C:\Users\simon\Documents\lab2etl\kaismart_etl\data\silver
  gold   -> C:\Users\simon\Documents\lab2etl\kaismart_etl\data\gold


---
# Parte 1. Extracción desde la base de datos

Se sigue el orden pedido en el enunciado. Los pasos 2 a 5 y el 7 están dentro de un único bloque `try / except / finally`
para que **la conexión se cierre siempre**, incluso si la consulta falla.

## Paso 1. Importar las librerías necesarias

`mysql.connector` establece la conexión con MySQL y `Error` es la excepción base de esa librería.

In [2]:
import mysql.connector
from mysql.connector import Error

## Pasos 2 a 5 y 7. Conectar, consultar, extraer, guardar en `df_ventas` y cerrar

Dentro del bloque, en orden:

* **Paso 2 – Conexión:** con los datos de `config.yaml`, la contraseña de `.env` y `connection_timeout=15` (si el servidor no responde en 15 s, falla en lugar de quedarse esperando).
* **Paso 3 – Consulta:** primero un `SELECT COUNT(*)` para saber cuántos registros hay en el origen y luego `SELECT *` sobre la tabla `ventas`.
* **Paso 4 – Extracción:** `fetchall()` trae todos los registros disponibles.
* **Paso 5 – DataFrame:** se construye `df_ventas` con los registros y los nombres de columna del cursor.
  (No se usa `pd.read_sql` con la conexión directa porque pandas emite un warning: solo soporta oficialmente SQLAlchemy.)
* **Validación:** un `assert` comprueba que lo extraído coincide con lo que hay en el origen.
* **Paso 7 – Cierre:** en `finally` se cierran el cursor y la conexión.

In [3]:
cfg_mysql = config["fuentes"]["mysql"]
tabla = cfg_mysql["tabla"]

# Se inicializan en None para que el bloque finally pueda cerrar solo lo que llegó a abrirse
conn = None
cursor = None

try:
    # ---- Paso 2. Crear la conexión con la base de datos
    conn = mysql.connector.connect(
        host=cfg_mysql["host"],
        port=cfg_mysql["port"],
        user=cfg_mysql["user"],
        password=os.getenv("DB_PASSWORD"),
        database=cfg_mysql["database"],
        connection_timeout=15,
    )
    cursor = conn.cursor()
    print(f"Conexión exitosa a la base de datos '{cfg_mysql['database']}' (MySQL {conn.server_info})")

    # ---- Paso 3. Consultar la tabla ventas
    cursor.execute(f"SELECT COUNT(*) FROM `{tabla}`")
    registros_origen = cursor.fetchone()[0]
    cursor.execute(f"SELECT * FROM `{tabla}`")

    # ---- Paso 4. Extraer todos los registros disponibles
    filas = cursor.fetchall()

    # ---- Paso 5. Almacenar el resultado en el DataFrame df_ventas
    df_ventas = pd.DataFrame(filas, columns=cursor.column_names)

    # ---- Validación de la extracción
    assert len(df_ventas) == registros_origen, (
        f"Se extrajeron {len(df_ventas)} registros pero el origen tiene {registros_origen}"
    )
    print(f"Registros en el origen (COUNT(*)):         {registros_origen:,}")
    print(f"Registros extraídos (df_ventas):           {len(df_ventas):,}")
    print(f"Registros esperados según el enunciado:    {cfg_mysql['registros_esperados']:,}")
    if len(df_ventas) != cfg_mysql["registros_esperados"]:
        print("ADVERTENCIA: la cantidad extraída no coincide con la esperada en el enunciado.")

except Error as e:
    print(f"Error de MySQL: {e}")
    raise

finally:
    # ---- Paso 7. Cerrar correctamente el cursor y la conexión
    if cursor is not None:
        cursor.close()
    if conn is not None and conn.is_connected():
        conn.close()
        print("Cursor y conexión con la base de datos cerrados correctamente.")

Conexión exitosa a la base de datos 'clientes' (MySQL 5.5.5-10.11.18-MariaDB-cll-lve)


Registros en el origen (COUNT(*)):         5,000
Registros extraídos (df_ventas):           5,000
Registros esperados según el enunciado:    5,000
Cursor y conexión con la base de datos cerrados correctamente.


## Paso 6. Comprobar que la extracción fue correcta

### 6.1 Dimensiones: `df_ventas.shape`

In [4]:
df_ventas.shape

(5000, 17)

### 6.2 Nombres de las columnas: `df_ventas.columns`

In [5]:
df_ventas.columns

Index(['id_venta', 'pedido_id', 'fecha_venta', 'id_cliente', 'ciudad', 'canal',
       'id_tienda', 'categoria', 'producto', 'cantidad', 'precio_unitario',
       'descuento_pct', 'valor_bruto', 'valor_descuento', 'valor_neto',
       'medio_pago', 'calificacion_cliente'],
      dtype='str')

### 6.3 Primeros cinco registros: `df_ventas.head()`

In [6]:
df_ventas.head()

,id_venta,pedido_id,fecha_venta,id_cliente,ciudad,canal,id_tienda,categoria,producto,cantidad,precio_unitario,descuento_pct,valor_bruto,valor_descuento,valor_neto,medio_pago,calificacion_cliente
0,1,PED000001,2026-01-18 22:06:15,CLI00011,Bogotá D.C.,Aplicación móvil,NaN,Deportes,Banda caminadora,1,2199900.00,10.00,2199900.00,219990.00,1979910.00,Tarjeta crédito,NaN
1,2,PED000002,2026-04-25 21:07:01,CLI00802,Cali,Página web,NaN,Deportes,Mancuernas ajustables,1,399900.00,10.00,399900.00,39990.00,359910.00,Tarjeta crédito,NaN
2,3,PED000003,2026-03-28 02:39:43,CLI00255,Medellín,Página web,NaN,Tecnología,Tablet 10 pulgadas,1,899900.00,10.00,899900.00,89990.00,809910.00,Tarjeta crédito,4.0
3,4,PED000004,2026-04-30 12:35:40,CLI00383,Medellín,Aplicación móvil,NaN,Electrodomésticos,Freidora de aire,2,472400.00,15.00,944800.00,141720.00,803080.00,PSE,5.0
4,5,PED000005,2026-04-04 00:18:12,CLI01286,Bogotá D.C.,Aplicación móvil,NaN,Deportes,Balón fútbol,1,119900.00,0.00,119900.00,0.00,119900.00,Tarjeta crédito,NaN


### 6.4 Muestra aleatoria de cinco registros: `df_ventas.sample(5, random_state=42)`

`sample(5)` elige cinco filas al azar. Con `random_state=42` se fija la semilla del generador aleatorio,
así que **la muestra es reproducible**: cada vez que se ejecute se obtienen las mismas cinco filas.

In [7]:
df_ventas.sample(5, random_state=42)

,id_venta,pedido_id,fecha_venta,id_cliente,ciudad,canal,id_tienda,categoria,producto,cantidad,precio_unitario,descuento_pct,valor_bruto,valor_descuento,valor_neto,medio_pago,calificacion_cliente
1501,1502,PED001502,2026-06-10 08:36:33,CLI02390,Bucaramanga,Tienda física,BGA-01,Tecnología,Monitor 24 pulgadas,2,713900.00,5.00,1427800.00,71390.00,1356410.00,Tarjeta débito,NaN
2586,2587,PED002587,2026-05-09 09:23:12,CLI01262,Bucaramanga,Aplicación móvil,NaN,Tecnología,Audífonos Bluetooth,2,149900.00,5.00,299800.00,14990.00,284810.00,Billetera digital,4.0
2653,2654,PED002654,2026-05-20 23:03:48,CLI01371,Cali,Aplicación móvil,NaN,Oficina,Base para portátil,1,112100.00,0.00,112100.00,0.00,112100.00,Tarjeta crédito,NaN
1055,1056,PED001056,2026-03-18 19:24:39,CLI00437,Medellín,Tienda física,MDE-01,Deportes,Bicicleta estática,2,1499900.00,0.00,2999800.00,0.00,2999800.00,Efectivo,5.0
705,706,PED000706,2026-06-06 19:01:06,CLI01135,Bogotá D.C.,Página web,NaN,Tecnología,Teclado mecánico,1,229900.00,5.00,229900.00,11495.00,218405.00,PSE,NaN


## Capa Bronze — ventas

Se guarda `df_ventas` en `data/bronze/ventas_bronze.csv` **exactamente como se extrajo** (sin transformar nada), sin índice y en UTF-8.

In [8]:
ruta_ventas_bronze = rutas["bronze"] / "ventas_bronze.csv"
df_ventas.to_csv(ruta_ventas_bronze, index=False, encoding="utf-8")
print(f"Bronze guardado: {ruta_ventas_bronze.relative_to(RAIZ)}  ({len(df_ventas):,} filas)")

Bronze guardado: data\bronze\ventas_bronze.csv  (5,000 filas)


---
# Parte 2. Extracción desde el archivo Excel

El archivo `kaismart_eventos_logisticos.xlsx` está en `data/raw/`. Se abre con `pd.ExcelFile` (motor `openpyxl`) dentro de un `with`,
de modo que el archivo se cierra automáticamente al terminar. Primero se listan las hojas y luego se lee la hoja configurada en `config.yaml`.
Si el archivo no existe, se lanza un `FileNotFoundError` con la ruta esperada.

In [9]:
cfg_excel = config["fuentes"]["excel"]
ruta_excel = rutas["raw"] / cfg_excel["archivo"]

if not ruta_excel.exists():
    raise FileNotFoundError(f"No se encontró el archivo Excel en la ruta esperada: {ruta_excel}")

with pd.ExcelFile(ruta_excel, engine="openpyxl") as xls:
    print("Hojas del archivo:", xls.sheet_names)
    df_logistica = pd.read_excel(xls, sheet_name=cfg_excel["hoja"])

print(f"Registros extraídos (df_logistica):        {len(df_logistica):,}")
print(f"Registros esperados según el enunciado:    {cfg_excel['registros_esperados']:,}")
if len(df_logistica) != cfg_excel["registros_esperados"]:
    print("ADVERTENCIA: la cantidad extraída no coincide con la esperada en el enunciado.")

Hojas del archivo: ['eventos_logisticos']


Registros extraídos (df_logistica):        50,000
Registros esperados según el enunciado:    50,000


## Comprobar que la extracción fue correcta

### Dimensiones: `df_logistica.shape`

In [10]:
df_logistica.shape

(50000, 14)

### Nombres de las columnas: `df_logistica.columns`

In [11]:
df_logistica.columns

Index(['evento_id', 'pedido_id', 'fecha_evento', 'estado_evento',
       'centro_logistico', 'ciudad_destino', 'transportadora', 'numero_guia',
       'fecha_prometida_entrega', 'tiempo_etapa_horas', 'costo_envio',
       'incidencia', 'observacion', 'Unnamed: 13'],
      dtype='str')

### Primeros cinco registros: `df_logistica.head()`

In [12]:
df_logistica.head()

,evento_id,pedido_id,fecha_evento,estado_evento,centro_logistico,ciudad_destino,transportadora,numero_guia,fecha_prometida_entrega,tiempo_etapa_horas,costo_envio,incidencia,observacion,Unnamed: 13
0,1,PED000001,2026-01-18 22:06:15,Pedido recibido,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,NaN,8900,NaN,NaN,"=AI("""")"
1,2,PED000001,2026-01-18 22:36:53,Pago aprobado,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,0.51,8900,NaN,NaN,NaN
2,3,PED000001,2026-01-18 23:31:22,Orden enviada a logística,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,0.91,8900,NaN,NaN,NaN
3,4,PED000001,2026-01-19 02:03:55,En alistamiento,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,2.54,8900,NaN,NaN,NaN
4,5,PED000001,2026-01-19 06:14:51,Empacado,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,4.18,8900,NaN,NaN,NaN


### Muestra aleatoria de cinco registros: `df_logistica.sample(5, random_state=42)`

In [13]:
df_logistica.sample(5, random_state=42)

,evento_id,pedido_id,fecha_evento,estado_evento,centro_logistico,ciudad_destino,transportadora,numero_guia,fecha_prometida_entrega,tiempo_etapa_horas,costo_envio,incidencia,observacion,Unnamed: 13
33553,33554,PED003356,2026-04-04 09:51:57,En alistamiento,CEDI Pereira,Pereira,NaN,NaN,2026-04-06 02:08:22,1.78,9900,NaN,Pedido procesado dentro de la ventana prevista.,NaN
9427,9428,PED000943,2026-04-14 11:09:42,En tránsito,CEDI Bucaramanga,Bucaramanga,Servientrega,KMS77388424,2026-04-15 16:24:14,9.56,8900,NaN,NaN,NaN
199,200,PED000020,2026-02-08 21:23:51,Entregado,CEDI Bogotá,Bogotá D.C.,TCC,KMS46254024,2026-02-09 11:26:46,5.13,13900,NaN,NaN,NaN
12447,12448,PED001245,2026-05-02 00:24:54,En tránsito,CEDI Cali,Cali,Inter Rapidísimo,KMS96024511,2026-05-03 00:03:57,9.38,15900,NaN,NaN,NaN
39489,39490,PED003949,2026-06-03 14:44:47,Entregado,CEDI Bogotá,Bogotá D.C.,Envia,KMS90495560,2026-06-03 02:22:34,9.09,17490,NaN,NaN,NaN


## Capa Bronze — logística

Se guarda `df_logistica` en `data/bronze/logistica_bronze.csv` sin transformar, sin índice y en UTF-8.

In [14]:
ruta_logistica_bronze = rutas["bronze"] / "logistica_bronze.csv"
df_logistica.to_csv(ruta_logistica_bronze, index=False, encoding="utf-8")
print(f"Bronze guardado: {ruta_logistica_bronze.relative_to(RAIZ)}  ({len(df_logistica):,} filas)")

Bronze guardado: data\bronze\logistica_bronze.csv  (50,000 filas)
